#  Fine-tuning con LoRA y QLoRA




```
Vamos a enseñarle a un modelo un **comportamiento**: responder consultas de atención al cliente de seguros de automóvil siguiendo siempre el mismo formato (una frase inicial, tres viñetas y la firma del departamento de atención al cliente).

No le enseñamos información nueva sobre las coberturas ni las condiciones de las pólizas: eso será responsabilidad
```

In [1]:
import torch

print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Modelo GPU:", torch.cuda.get_device_name(0))

GPU disponible: True
Modelo GPU: Tesla T4


In [3]:
# 0 · Instalar las librerias (en Colab ya vienen torch y transformers; anadimos el resto)
import importlib.util, importlib.metadata, subprocess, sys
faltan = [p for p in ("peft", "bitsandbytes", "datasets", "accelerate") if importlib.util.find_spec(p) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=False)
# Colab trae de serie una version antigua de torchao que hace fallar a peft al fusionar el adapter. No la usamos: fuera.
try:
    v = importlib.metadata.version("torchao")
    if tuple(int(x) for x in v.split(".")[:2]) < (0, 16):
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
        print(f"torchao {v} desinstalado (incompatible con peft)")
except importlib.metadata.PackageNotFoundError:
    pass
print("Librerias listas")

torchao 0.10.0 desinstalado (incompatible con peft)
Librerias listas


## Paso 1 · Preparar el dataset

10 ejemplos de pregunta-respuesta sobre seguros de automóvil, **todos con exactamente el mismo formato**. La calidad y la consistencia del dataset son fundamentales para que el modelo aprenda el comportamiento deseado.

Los ejemplos se guardan en formato **JSONL**: una línea por ejemplo. Las 6 preguntas de prueba no se utilizan para entrenar: sirven para comprobar si el modelo ha aprendido el formato ante consultas que no ha visto durante el entrenamiento.

In [1]:

EJEMPLOS = [
    {
        "instruccion": "¿Cómo comunico un accidente de tráfico?",
        "respuesta": "Un accidente de tráfico debe comunicarse a la aseguradora lo antes posible\n- Facilite la fecha, el lugar y una descripción de lo sucedido.\n- Aporte los datos de los vehículos y de las personas implicadas.\n- Adjunte fotografías y el parte amistoso si dispone de ellos.\n— Departamento de Atención al Cliente"
    },
    {
        "instruccion": "He tenido un golpe con el coche, ¿qué hago?",
        "respuesta": "Lo primero es garantizar la seguridad y documentar el incidente\n- Compruebe si hay personas heridas y contacte con emergencias si es necesario.\n- Recopile los datos de los vehículos y de los conductores implicados.\n- Comunique el accidente a su aseguradora siguiendo los canales disponibles.\n— Departamento de Atención al Cliente"
    },
    {
        "instruccion": "¿Cómo puedo solicitar asistencia en carretera?",
        "respuesta": "Puede solicitar asistencia en carretera a través de los canales de su aseguradora\n- Consulte el teléfono de asistencia indicado en su documentación.\n- Facilite su ubicación y los datos de identificación de su vehículo.\n- Describa la incidencia para que puedan valorar la asistencia correspondiente.\n— Departamento de Atención al Cliente"
    },
    {
        "instruccion": "Necesito una copia de mi póliza",
        "respuesta": "Puede solicitar una copia de su póliza a la compañía aseguradora\n- Acceda al área de cliente si dispone de este servicio.\n- Identifique su contrato con los datos solicitados.\n- Si no encuentra el documento, contacte con atención al cliente.\n— Departamento de Atención al Cliente"
    },
    {
        "instruccion": "¿Cómo puedo comunicar un cambio de domicilio?",
        "respuesta": "Es importante mantener actualizados sus datos personales\n- Contacte con su aseguradora a través de los canales habilitados.\n- Indique su nueva dirección y los datos necesarios para identificar su póliza.\n- Solicite confirmación de que la modificación ha quedado registrada.\n— Departamento de Atención al Cliente"
    },
    {
        "instruccion": "Me han cobrado un recibo que no reconozco",
        "respuesta": "Si no reconoce un recibo, le recomendamos solicitar una revisión\n- Compruebe el importe y la fecha del cargo bancario.\n- Identifique la póliza o el contrato al que podría corresponder.\n- Contacte con atención al cliente y solicite una aclaración del cobro.\n— Departamento de Atención al Cliente"
    },
    {
        "instruccion": "¿Qué documentación necesito para declarar un siniestro?",
        "respuesta": "La documentación depende del tipo de siniestro comunicado\n- Prepare sus datos personales y el número de póliza.\n- Reúna fotografías, informes y documentos relacionados con el incidente.\n- Consulte con la aseguradora si necesita aportar documentación adicional.\n— Departamento de Atención al Cliente"
    },
    {
        "instruccion": "Tengo una luna rota, ¿cómo lo gestiono?",
        "respuesta": "Para gestionar una incidencia relacionada con las lunas, contacte con su aseguradora\n- Facilite el número de póliza y los datos de su vehículo.\n- Describa los daños y aporte fotografías si se las solicitan.\n- Consulte las condiciones de su contrato y el procedimiento de reparación.\n— Departamento de Atención al Cliente"
    },
    {
        "instruccion": "Quiero dar de baja mi seguro de coche",
        "respuesta": "Para solicitar la baja de su seguro, debe contactar con la aseguradora\n- Facilite los datos de identificación de su póliza.\n- Consulte los plazos y las condiciones aplicables a la cancelación.\n- Solicite una confirmación escrita de la recepción de su petición.\n— Departamento de Atención al Cliente"
    },
    {
        "instruccion": "¿Cómo puedo consultar las coberturas de mi seguro?",
        "respuesta": "Las coberturas contratadas se detallan en las condiciones de su póliza\n- Consulte las condiciones particulares y generales del contrato.\n- Identifique las garantías incluidas y sus posibles limitaciones.\n- Si necesita aclaraciones, contacte con su aseguradora indicando la cobertura consultada.\n— Departamento de Atención al Cliente"
    }
]

PRUEBA = [
    "Me han dado un golpe y el otro conductor se ha marchado",
    "¿Puedo elegir el taller donde reparar mi vehículo?",
    "No encuentro el justificante del último recibo",
    "¿Qué debo hacer si me roban el coche?",
    "Quiero saber si tengo cobertura de vehículo de sustitución",
    "¿Cómo puedo presentar una reclamación a la aseguradora?"
]

import json

with open("seguros_formato.jsonl", "w", encoding="utf-8") as f:
    for e in EJEMPLOS:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")

print(f"{len(EJEMPLOS)} ejemplos guardados en seguros_formato.jsonl · {len(PRUEBA)} preguntas de prueba aparte\n")

for e in EJEMPLOS[:2]:
    print("PREGUNTA :", e["instruccion"])
    print("RESPUESTA:", e["respuesta"].replace("\n", "\n           "), "\n")


10 ejemplos guardados en seguros_formato.jsonl · 6 preguntas de prueba aparte

PREGUNTA : ¿Cómo comunico un accidente de tráfico?
RESPUESTA: Un accidente de tráfico debe comunicarse a la aseguradora lo antes posible
           - Facilite la fecha, el lugar y una descripción de lo sucedido.
           - Aporte los datos de los vehículos y de las personas implicadas.
           - Adjunte fotografías y el parte amistoso si dispone de ellos.
           — Departamento de Atención al Cliente 

PREGUNTA : He tenido un golpe con el coche, ¿qué hago?
RESPUESTA: Lo primero es garantizar la seguridad y documentar el incidente
           - Compruebe si hay personas heridas y contacte con emergencias si es necesario.
           - Recopile los datos de los vehículos y de los conductores implicados.
           - Comunique el accidente a su aseguradora siguiendo los canales disponibles.
           — Departamento de Atención al Cliente 



In [2]:
# El formato que queremos ensenar, escrito como una comprobacion con codigo
FIRMA = "— Departamento de Atención al Cliente"

def cumple_formato(texto: str) -> bool:
    lineas = [l.strip() for l in texto.strip().split("\n") if l.strip()]
    return (
        len(lineas) == 5
        and not lineas[0].startswith("-")
        and all(l.startswith("- ") for l in lineas[1:4])
        and lineas[4] == FIRMA
    )

print("Los 10 ejemplos cumplen el formato:",
      all(cumple_formato(e["respuesta"]) for e in EJEMPLOS))

Los 10 ejemplos cumplen el formato: True


## Paso 2 · Cargar el modelo en 4 bits (QLoRA)

Utilizamos un modelo pequeño de 1.500 millones de parámetros (Qwen2.5-1.5B-Instruct), **cuantizado a 4 bits** con bitsandbytes. Esta técnica reduce considerablemente la memoria necesaria para cargar y entrenar el modelo.

El entrenamiento se realiza en una GPU Tesla T4 de Google Colab.

In [4]:
import time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU = torch.cuda.is_available()
MODELO = "Qwen/Qwen2.5-1.5B-Instruct" if GPU else "HuggingFaceTB/SmolLM2-135M-Instruct"
print("GPU:", torch.cuda.get_device_name(0) if GPU else "no hay GPU -> MODO PRUEBA (modelo diminuto, 2 pasos)")

tok = AutoTokenizer.from_pretrained(MODELO)
tok.pad_token = tok.pad_token or tok.eos_token
t0 = time.time()
if GPU:
    from transformers import BitsAndBytesConfig
    config_4bit = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                     bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    modelo = AutoModelForCausalLM.from_pretrained(MODELO, quantization_config=config_4bit, device_map="auto")
else:
    modelo = AutoModelForCausalLM.from_pretrained(MODELO)
print(f"Modelo: {MODELO}")
print(f"Cargado en {time.time() - t0:.0f} s · ocupa {modelo.get_memory_footprint() / 1e9:.2f} GB en memoria")
if GPU:
    print(f"Memoria de la GPU en uso: {torch.cuda.memory_allocated() / 1e9:.2f} GB de {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo: Qwen/Qwen2.5-1.5B-Instruct
Cargado en 47 s · ocupa 1.12 GB en memoria
Memoria de la GPU en uso: 1.15 GB de 15.6 GB


### Variacion · ¿Cuanto ahorran los 4 bits?
Cargamos el mismo modelo en 16 bits solo para medirlo, y lo borramos.

In [ ]:
if GPU:
    m16 = AutoModelForCausalLM.from_pretrained(MODELO, torch_dtype=torch.float16, device_map="auto")
    gb16, gb4 = m16.get_memory_footprint() / 1e9, modelo.get_memory_footprint() / 1e9
    print(f"16 bits: {gb16:.2f} GB · 4 bits: {gb4:.2f} GB · {gb16 / gb4:.1f} veces menos")
    del m16; torch.cuda.empty_cache()
else:
    print("(solo con GPU)")

(solo con GPU)


## Paso 3 · Entrenar el adapter LoRA

LoRA congela los pesos del modelo base y añade unas matrices pequeñas (el *adapter*) que son las que se entrenan. De esta forma, adaptamos el comportamiento del modelo sin tener que entrenar todos sus parámetros.

Dos parámetros importantes son el **rank** (r, capacidad de adaptación) y **alpha** (factor de escalado). En este ejercicio utilizamos r=8 y alpha=16, con LoRA aplicado a todas las capas lineales.

Configuración del entrenamiento: **10 épocas**, batch size 4 y learning rate 3e-4.

In [5]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Antes de tocar nada, guardamos lo que contesta el modelo BASE a las preguntas de prueba
def responder(m, pregunta, max_tokens=None):
    # return_dict=True: funciona igual con transformers 4.x y 5.x (en la 5.x ya no devuelve solo los ids)
    enc = tok.apply_chat_template([{"role": "user", "content": pregunta}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(m.device)
    with torch.no_grad():
        out = m.generate(**enc, max_new_tokens=max_tokens or (160 if GPU else 40), do_sample=False,
                         pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

preguntas = PRUEBA if GPU else PRUEBA[:2]
antes = {p: responder(modelo, p) for p in preguntas}
print(f"Respuestas del modelo base guardadas ({len(antes)})")

if GPU:
    modelo = prepare_model_for_kbit_training(modelo)
config_lora = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                         target_modules="all-linear")     # atencion (q, k, v, o) y MLP (gate, up, down)
modelo = get_peft_model(modelo, config_lora)
modelo.print_trainable_parameters()

Respuestas del modelo base guardadas (6)
trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


### Variacion · ¿Cuanto pesa el adapter segun el rank?
Contamos los parametros entrenables con distintos rank, sin entrenar.

In [ ]:
# Los parametros del adapter crecen en proporcion al rank (cada matriz tiene r filas o r columnas)
entrenables, todos = modelo.get_nb_trainable_parameters()     # peft ya descuenta el empaquetado de 4 bits
base = todos - entrenables
print(f"{'rank':>5}{'parametros entrenables':>25}{'% del modelo':>15}{'adapter en disco':>19}")
for r in (4, 8, 16, 64):
    n = entrenables * r // 8
    print(f"{r:>5}{n:>25,}{100 * n / (base + n):>14.3f}%{n * 4 / 1e6:>16.1f} MB")

 rank   parametros entrenables   % del modelo   adapter en disco
    4                1,221,120         0.900%             4.9 MB
    8                2,442,240         1.783%             9.8 MB
   16                4,884,480         3.504%            19.5 MB
   64               19,537,920        12.683%            78.2 MB


In [7]:
# Tokenizar: el modelo solo aprende de la RESPUESTA (las etiquetas de la pregunta se ponen a -100)
from datasets import Dataset

def preparar(e):
    prompt = tok.apply_chat_template([{"role": "user", "content": e["instruccion"]}], add_generation_prompt=True, tokenize=False)
    completo = tok.apply_chat_template([{"role": "user", "content": e["instruccion"]},
                                        {"role": "assistant", "content": e["respuesta"]}], tokenize=False)
    n_prompt = len(tok(prompt, add_special_tokens=False)["input_ids"])
    ids = tok(completo, add_special_tokens=False)["input_ids"]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": [-100] * n_prompt + ids[n_prompt:]}

ds = Dataset.from_list(EJEMPLOS).map(preparar, remove_columns=["instruccion", "respuesta"])
print(f"{len(ds)} ejemplos tokenizados · el mas largo tiene {max(len(x) for x in ds['input_ids'])} tokens")

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

10 ejemplos tokenizados · el mas largo tiene 126 tokens


In [8]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

args = TrainingArguments(output_dir="entrenamiento", num_train_epochs=10, per_device_train_batch_size=4,
                         learning_rate=3e-4, logging_steps=4 if GPU else 1, save_strategy="no", report_to="none",
                         fp16=GPU, max_steps=-1 if GPU else 2, optim="adamw_torch")
trainer = Trainer(model=modelo, args=args, train_dataset=ds,
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
t0 = time.time()
trainer.train()
print(f"\nEntrenamiento: {time.time() - t0:.0f} s")
print(f"{'paso':>6}{'epoca':>8}{'loss':>9}")
for h in trainer.state.log_history:
    if "loss" in h:
        print(f"{h['step']:>6}{h['epoch']:>8.1f}{h['loss']:>9.3f}")

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
4,2.151525
8,1.233480
12,0.835832
16,0.475276
20,0.278230
24,0.138749
28,0.077797



Entrenamiento: 20 s
  paso   epoca     loss
     4     1.3    2.152
     8     2.7    1.233
    12     4.0    0.836
    16     5.3    0.475
    20     6.7    0.278
    24     8.0    0.139
    28     9.3    0.078


## Paso 4 · Probar: antes frente a después

Utilizamos las mismas 6 preguntas de prueba para comparar las respuestas del modelo base y del modelo ajustado. Medimos mediante código cuántas respuestas cumplen el formato establecido para atención al cliente de seguros.

En nuestro experimento, el modelo base obtuvo **0 de 6** respuestas que cumplían el formato, mientras que el modelo ajustado obtuvo **4 de 6**.

In [10]:

modelo.eval()

despues = {p: responder(modelo, p) for p in preguntas}

ok_antes = sum(cumple_formato(antes[p]) for p in preguntas)
ok_despues = sum(cumple_formato(despues[p]) for p in preguntas)

print(
    f"Cumplen el formato de seguros · "
    f"BASE: {ok_antes}/{len(preguntas)} · "
    f"AJUSTADO: {ok_despues}/{len(preguntas)}\n"
)

for p in preguntas:
    print(
        f"  {'OK' if cumple_formato(antes[p]) else '--'} base · "
        f"{'OK' if cumple_formato(despues[p]) else '--'} ajustado · {p}"
    )

for p in preguntas[:2]:
    print("=" * 90)
    print("PREGUNTA:", p)
    print("-- BASE --")
    print(antes[p])
    print("-- AJUSTADO con LoRA --")
    print(despues[p])


Cumplen el formato de seguros · BASE: 0/6 · AJUSTADO: 4/6

  -- base · OK ajustado · Me han dado un golpe y el otro conductor se ha marchado
  -- base · -- ajustado · ¿Puedo elegir el taller donde reparar mi vehículo?
  -- base · OK ajustado · No encuentro el justificante del último recibo
  -- base · OK ajustado · ¿Qué debo hacer si me roban el coche?
  -- base · OK ajustado · Quiero saber si tengo cobertura de vehículo de sustitución
  -- base · -- ajustado · ¿Cómo puedo presentar una reclamación a la aseguradora?
PREGUNTA: Me han dado un golpe y el otro conductor se ha marchado
-- BASE --
I'm sorry to hear that you've been hit and the other driver has left. It's important to ensure your safety in such situations. If possible, try to call for help or seek medical attention if necessary. Please take care of yourself and stay safe.
-- AJUSTADO con LoRA --
Lo primero es garantizar la seguridad de los ocupantes
- Compruebe si hay personas heridas y contacte con emergencias si es necesari

### Variación · El fine-tuning enseña la FORMA, no los DATOS

Probamos el modelo con una consulta sobre la rotura de una luna del vehículo, un supuesto relacionado con las coberturas del seguro.

El modelo ajustado reproduce la estructura de atención al cliente, pero puede generar información imprecisa sobre las condiciones o las prestaciones. Para responder con información fiable será necesario recuperar los datos de la póliza mediante RAG.

In [11]:

dato = "¿Qué debo hacer si tengo una luna rota en mi vehículo?"

print("PREGUNTA:", dato, "\n")
print(responder(modelo, dato))

print("\n(Comprobar si la respuesta sigue el formato de atención al cliente de seguros.)")


PREGUNTA: ¿Qué debo hacer si tengo una luna rota en mi vehículo? 

Si tiene una lunar en su vehículo, debe contactar con la aseguradora
- Facilite los datos de su póliza y la descripción de la incidencia.
- Consulte las condiciones aplicables a las lunas detectadas.
- Puede solicitar la reparación correspondiente o un abono económico.
— Departamento de Atención al Cliente

(Comprobar si la respuesta sigue el formato de atención al cliente de seguros.)


### Fusionar (merge) y guardar

El adapter LoRA se guarda por separado y también se puede **fusionar** con el modelo base para obtener un modelo completo que pueda utilizarse posteriormente.

En este ejercicio hemos guardado el adaptador en `adapter_seguros/` y el modelo fusionado en `modelo_seguros_fusionado/`.

In [12]:

import os, shutil
from peft import PeftModel

# 1. Guardar el adaptador LoRA
modelo.save_pretrained("adapter_seguros")
tok.save_pretrained("adapter_seguros")

mb = os.path.getsize(
    os.path.join("adapter_seguros", "adapter_model.safetensors")
) / 1e6

print(f"Adapter guardado en adapter_seguros/ · sus pesos ocupan {mb:.1f} MB")

# 2. Cargar el modelo base y fusionar el adaptador
base16 = AutoModelForCausalLM.from_pretrained(
    MODELO,
    torch_dtype=torch.float16 if GPU else torch.float32,
    device_map="auto" if GPU else None
)

fusionado = PeftModel.from_pretrained(
    base16, "adapter_seguros"
).merge_and_unload()

# 3. Guardar el modelo fusionado
fusionado.save_pretrained("modelo_seguros_fusionado")
tok.save_pretrained("modelo_seguros_fusionado")

gb = sum(
    os.path.getsize(os.path.join("modelo_seguros_fusionado", f))
    for f in os.listdir("modelo_seguros_fusionado")
) / 1e9

print(f"Modelo fusionado guardado en modelo_seguros_fusionado/ · {gb:.2f} GB")

# 4. Comprobar el formato de respuesta de seguros
print(
    "El modelo fusionado cumple el formato de seguros:",
    cumple_formato(responder(fusionado, preguntas[0]))
)

# 5. Crear ZIP del adaptador
shutil.make_archive("adapter_seguros", "zip", "adapter_seguros")
print("adapter_seguros.zip listo para descargar")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Adapter guardado en adapter_seguros/ · sus pesos ocupan 37.0 MB


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo fusionado guardado en modelo_seguros_fusionado/ · 3.10 GB
El modelo fusionado cumple el formato de seguros: True
adapter_seguros.zip listo para descargar


## Paso 5 · Cierre: ¿fine-tuning o RAG?

- **Fine-tuning** cambia CÓMO responde el modelo: formato, tono y estilo. En nuestro experimento, el modelo ajustado mejoró el cumplimiento del formato de 0/6 a 4/6.
- **RAG** permite consultar QUÉ sabe: información concreta de los seguros, como coberturas, exclusiones, procedimientos y condiciones de las pólizas.
- En producción se **combinan**: el adapter LoRA aporta el formato y el estilo de respuesta, mientras que el RAG recupera la información documental necesaria para fundamentar las respuestas.

In [13]:

# Resumen de la ejecución del proyecto de seguros

import json

resumen = {
    "modelo": MODELO,
    "gpu": torch.cuda.get_device_name(0) if GPU else None,
    "ejemplos_entrenamiento": len(EJEMPLOS),
    "preguntas_prueba": len(preguntas),
    "formato_antes": f"{ok_antes}/{len(preguntas)}",
    "formato_despues": f"{ok_despues}/{len(preguntas)}",
    "adapter_mb": round(mb, 1),
    "fusionado_gb": round(gb, 2)
}

json.dump(
    {"resumen": resumen, "antes": antes, "despues": despues},
    open("resultados_seguros.json", "w", encoding="utf-8"),
    ensure_ascii=False,
    indent=1
)

print(json.dumps(resumen, ensure_ascii=False, indent=1))


{
 "modelo": "Qwen/Qwen2.5-1.5B-Instruct",
 "gpu": "Tesla T4",
 "ejemplos_entrenamiento": 10,
 "preguntas_prueba": 6,
 "formato_antes": "0/6",
 "formato_despues": "4/6",
 "adapter_mb": 37.0,
 "fusionado_gb": 3.1
}
